# Week 7 Assignment - Delta Lake MERGE for Incremental Data Processing

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

Loading the customer master dataset

In [0]:
#reading the customer master csv file
master_df = spark.read.csv("/Volumes/workspace/default/week-7/customer_master.csv", header=True, inferSchema=True)
master_df.show(5)

+-----------+-------------+--------+-------------+-----------+--------------+-----------+-------+
|Customer_ID|Customer_Name| Segment|      Country|       City|         State|Postal_Code| Region|
+-----------+-------------+--------+-------------+-----------+--------------+-----------+-------+
|   AA-10315|   Alex Avila|Consumer|United States|Minneapolis|     Minnesota|      55407|Central|
|   AA-10375| Allen Armold|Consumer|United States|       Mesa|       Arizona|      85204|   West|
|   AA-10480| Andrew Allen|Consumer|United States|    Concord|North Carolina|      28027|  South|
|   AA-10645|Anna Andreadi|Consumer|United States|    Chester|  Pennsylvania|      19013|   East|
|   AB-10015|Aaron Bergman|Consumer|United States|    Seattle|    Washington|      98103|   West|
+-----------+-------------+--------+-------------+-----------+--------------+-----------+-------+
only showing top 5 rows


Data Exploration

In [0]:
#schema of the data
master_df.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)



In [0]:
#number of records and columns
print("Total Records:", master_df.count())
print("Total Columns:", len(master_df.columns))

Total Records: 793
Total Columns: 8


In [0]:
#column names
master_df.columns

['Customer_ID',
 'Customer_Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal_Code',
 'Region']

Data Cleaning

In [0]:
#checking for null values in each column
master_df.select([count(when(col(c).isNull(), 1)).alias(c) for c in master_df.columns]).show()

+-----------+-------------+-------+-------+----+-----+-----------+------+
|Customer_ID|Customer_Name|Segment|Country|City|State|Postal_Code|Region|
+-----------+-------------+-------+-------+----+-----+-----------+------+
|          0|            0|      0|      0|   0|    0|          0|     0|
+-----------+-------------+-------+-------+----+-----+-----------+------+



In [0]:
#removing duplicates based on Customer_ID
master_df = master_df.dropDuplicates(["Customer_ID"])
master_df.count()

793

In [0]:
#filling any null values with defaults (if any exist)
master_df = master_df.fillna({
    "Segment": "Consumer",
    "Country": "United States",
    "City": "Unknown",
    "State": "Unknown",
    "Postal_Code": "00000",
    "Region": "Unknown"
})


In [0]:
#verifying nulls are handled
master_df.select([count(when(col(c).isNull(), 1)).alias(c) for c in master_df.columns]).show()

+-----------+-------------+-------+-------+----+-----+-----------+------+
|Customer_ID|Customer_Name|Segment|Country|City|State|Postal_Code|Region|
+-----------+-------------+-------+-------+----+-----+-----------+------+
|          0|            0|      0|      0|   0|    0|          0|     0|
+-----------+-------------+-------+-------+----+-----+-----------+------+



Creating Delta Table for SCD Type 1

In [0]:
#saving as managed delta table in unity catalog
master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.customer_master_delta")

In [0]:
#reading back the delta table to verify
display(spark.table("workspace.default.customer_master_delta"))

Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region
AA-10645,Anna Andreadi,Consumer,United States,Chester,Pennsylvania,19013,East
AB-10600,Ann Blume,Corporate,United States,Tucson,Arizona,85705,West
AG-10270,Alejandro Grove,Consumer,United States,West Jordan,Utah,84084,West
BD-11770,Bryan Davis,Consumer,United States,Houston,Texas,77070,Central
BF-10975,Barbara Fisher,Corporate,United States,Charlotte,North Carolina,28205,South
BG-11035,Barry Gonzalez,Consumer,United States,Monroe,Louisiana,71203,South
CK-12760,Cyma Kinney,Corporate,United States,Linden,New Jersey,7036,East
CL-11890,Carl Ludwig,Consumer,United States,Everett,Massachusetts,2149,East
DJ-13420,Denny Joy,Corporate,United States,Warner Robins,Georgia,31088,South
EB-13930,Eric Barreto,Consumer,United States,San Francisco,California,94110,West


Loading the incremental dataset

In [0]:
#reading customer incremental csv from unity catalog volume
incremental_df = spark.read.csv("/Volumes/workspace/default/week-7/customer_incremental.csv", header=True, inferSchema=True)
display(incremental_df)

Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region
AA-10315,Alex Avila,Corporate,United States,Hyderabad,Minnesota,55407,Central
AA-10375,Allen Armold,Home Office,United States,Mumbai,Arizona,85204,West
AA-10480,Andrew Allen,Consumer,United States,Bangalore,North Carolina,28027,South
SU-00001,Surya Intern,Consumer,India,Hyderabad,Telangana,500001,South
SU-00002,Test Customer,Corporate,India,Bangalore,Karnataka,560001,South


In [0]:
#checking incremental record count
incremental_df.count()

5

Applying SCD Type 1 (MERGE Operation)

In [0]:
#loading the delta table
delta_table = DeltaTable.forName(spark, "workspace.default.customer_master_delta")

In [0]:
#count before merge
print("before merge count:", spark.table("workspace.default.customer_master_delta").count())

before merge count: 793


In [0]:
#performing merge to update existing and insert new records based on Customer_ID
delta_table.alias("target") \
    .merge(
        incremental_df.alias("source"),
        "target.Customer_ID = source.Customer_ID"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#count after merge
print("after merge count:", spark.table("workspace.default.customer_master_delta").count())
#should be master records + 2 new records (793 + 2 = 795)

after merge count: 795


Validating SCD Type 1 Results

In [0]:
final_df = spark.table("workspace.default.customer_master_delta")

In [0]:
#checking for duplicates on Customer_ID (should be none)
final_df.groupBy("Customer_ID") \
    .agg(count("*").alias("cnt")) \
    .filter("cnt > 1") \
    .show()

+-----------+---+
|Customer_ID|cnt|
+-----------+---+
+-----------+---+



In [0]:
#verifying updates (first 3 customers should have updated City and Segment)
#customer CG-12520 should be Hyderabad / Corporate
#customer DV-13045 should be Mumbai / Home Office
#customer SO-20335 should be Bangalore / Consumer
display(final_df.filter(col("Customer_ID").isin("CG-12520", "DV-13045", "SO-20335")))

Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region
DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West
SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South
CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South


In [0]:
#verifying new records got inserted
display(final_df.filter(col("Customer_ID").isin("SU-00001", "SU-00002")))

Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region
SU-00001,Surya Intern,Consumer,India,Hyderabad,Telangana,500001,South
SU-00002,Test Customer,Corporate,India,Bangalore,Karnataka,560001,South


Implementing SCD Type 2 (History Tracking)

for scd type 2 we need to track history by adding is_current, start_date and end_date columns

In [0]:
#preparing base master data for scd2 by adding tracking columns
#set start_date to current_date, end_date to null, is_current to true
master_scd2_df = master_df \
    .withColumn("is_current", lit(True)) \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date"))

#saving as delta table for scd2
master_scd2_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.customer_scd2_delta")

In [0]:
#verifying the scd2 base table
display(spark.table("workspace.default.customer_scd2_delta"))

Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,is_current,start_date,end_date
AA-10645,Anna Andreadi,Consumer,United States,Chester,Pennsylvania,19013,East,true,2026-07-12,null
AB-10600,Ann Blume,Corporate,United States,Tucson,Arizona,85705,West,true,2026-07-12,null
AG-10270,Alejandro Grove,Consumer,United States,West Jordan,Utah,84084,West,true,2026-07-12,null
BD-11770,Bryan Davis,Consumer,United States,Houston,Texas,77070,Central,true,2026-07-12,null
BF-10975,Barbara Fisher,Corporate,United States,Charlotte,North Carolina,28205,South,true,2026-07-12,null
BG-11035,Barry Gonzalez,Consumer,United States,Monroe,Louisiana,71203,South,true,2026-07-12,null
CK-12760,Cyma Kinney,Corporate,United States,Linden,New Jersey,7036,East,true,2026-07-12,null
CL-11890,Carl Ludwig,Consumer,United States,Everett,Massachusetts,2149,East,true,2026-07-12,null
DJ-13420,Denny Joy,Corporate,United States,Warner Robins,Georgia,31088,South,true,2026-07-12,null
EB-13930,Eric Barreto,Consumer,United States,San Francisco,California,94110,West,true,2026-07-12,null


In [0]:
#performing scd type 2 merge
scd2_table = DeltaTable.forName(spark, "workspace.default.customer_scd2_delta")

#find updates: rows that match on Customer_ID, are currently active (is_current = True), and have changed values
#since we want to update City and Segment, we check if they are different
target_df = spark.table("workspace.default.customer_scd2_delta").filter(col("is_current") == True)

updates_df = incremental_df.join(target_df, "Customer_ID") \
    .filter(
        (incremental_df.City != target_df.City) | 
        (incremental_df.Segment != target_df.Segment)
    ) \
    .select(incremental_df["Customer_ID"])

#expire old records: set is_current = false, end_date = current_date
#we merge updates into target and when matched, update history columns
scd2_table.alias("target") \
    .merge(
        updates_df.alias("updates"),
        "target.Customer_ID = updates.Customer_ID AND target.is_current = true"
    ) \
    .whenMatchedUpdate(set={
        "is_current": "false",
        "end_date": "current_date()"
    }) \
    .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#step 2: insert both brand new records and new active versions of updated records
#we prepare new records with tracking columns (is_current = True, start_date = current_date, end_date = null)
new_records_to_insert = incremental_df \
    .withColumn("is_current", lit(True)) \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date"))

#insert into delta table
scd2_table.alias("target") \
    .merge(
        new_records_to_insert.alias("source"),
        "target.Customer_ID = source.Customer_ID AND target.is_current = true"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Validating SCD Type 2 Results

In [0]:
final_scd2_df = spark.table("workspace.default.customer_scd2_delta")

In [0]:
#checking total rows in scd2 table
final_scd2_df.count()
#should be master records + 2 new records + 3 updated history records (793 + 2 + 3 = 798)

798

In [0]:
#verifying history for updated customers (should have two rows: one active, one inactive)
display(final_scd2_df.filter(col("Customer_ID").isin("CG-12520", "DV-13045", "SO-20335")).orderBy("Customer_ID", "start_date"))

Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,is_current,start_date,end_date
CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,true,2026-07-12,null
DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,true,2026-07-12,null
SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,true,2026-07-12,null


In [0]:
#verifying new customer records (should have one active row)
display(final_scd2_df.filter(col("Customer_ID").isin("SU-00001", "SU-00002")))

Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,is_current,start_date,end_date
SU-00001,Surya Intern,Consumer,India,Hyderabad,Telangana,500001,South,true,2026-07-12,null
SU-00002,Test Customer,Corporate,India,Bangalore,Karnataka,560001,South,true,2026-07-12,null


Delta Table History

In [0]:
#checking version history for scd2 delta table
scd2_table = DeltaTable.forName(spark, "workspace.default.customer_scd2_delta")
display(scd2_table.history())

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-07-12T15:42:03.000Z,76794227397085,surya.rithwik2005@gmail.com,MERGE,"Map(predicate -> [""((Customer_ID#17464 = Customer_ID#15420) AND is_current#17472)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(844671071626352),eaf7cf9c-8255-4d90-8861-28c2407d6cf9,0712-152707-ddj59gcu-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3448, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1881, materializeSourceTimeMs -> 11, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1806)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-12T15:41:59.000Z,76794227397085,surya.rithwik2005@gmail.com,MERGE,"Map(predicate -> [""((Customer_ID#16853 = Customer_ID#15420) AND is_current#16861)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(844671071626352),8c4611ca-856b-4e9b-9cfa-e37581a5cd79,0712-152707-ddj59gcu-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3318, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 3, executionTimeMs -> 5136, materializeSourceTimeMs -> 664, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1632, numTargetRowsUpdated -> 3, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2770)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-12T15:41:50.000Z,76794227397085,surya.rithwik2005@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(844671071626352),b730efd5-8ab9-410b-bcee-89b946f867c9,0712-152707-ddj59gcu-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 793, numOutputBytes -> 17758)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


Final Summary

In [0]:
#segment wise active customer counts
final_scd2_df.filter(col("is_current") == True) \
    .groupBy("Segment") \
    .agg(
        count("*").alias("active_customers")
    ) \
    .orderBy("Segment") \
    .show()

+-----------+----------------+
|    Segment|active_customers|
+-----------+----------------+
|   Consumer|             408|
|  Corporate|             238|
|Home Office|             149|
+-----------+----------------+



In [0]:
#region wise distribution of active customers
final_scd2_df.filter(col("is_current") == True) \
    .groupBy("Region") \
    .agg(
        count("*").alias("active_customers")
    ) \
    .orderBy("Region") \
    .show()

+-------+----------------+
| Region|active_customers|
+-------+----------------+
|Central|             184|
|   East|             220|
|  South|             136|
|   West|             255|
+-------+----------------+



## Observations

- loaded the customer master dataset which has 793 records with 8 columns from Unity Catalog Volume
- cleaned the data by checking nulls and removing duplicates on Customer_ID
- saved the cleaned master data as a managed delta table workspace.default.customer_master_delta
- loaded the incremental customer dataset which has 5 records (3 updates, 2 new inserts)
- performed scd type 1 merge using Customer_ID as primary key to overwrite existing values and insert new rows
- validated scd type 1 results - total rows is 795 and updates reflected properly
- implemented scd type 2 by adding history tracking columns: is_current, start_date and end_date
- performed scd type 2 merge - first expired old records (is_current=false, end_date=current_date) and then inserted new active records
- validated scd type 2 - total rows is 798 (793 + 2 inserts + 3 history rows) and verified that updated customers have both active and inactive records in history
- checked delta table history which lists the write and merge operations